# 🌳 Decision Trees Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **Decision Trees**! In this notebook, we will:
1. Generate a synthetic 3-class dataset based on the bounding box case study (`valve`, `flange`, `small-fitting`).
2. Train a **Decision Tree Classifier** using `scikit-learn`.
3. Visualize the axis-aligned orthogonal decision boundaries.
4. Visualize the tree structure itself.
5. Implement a **Decision Tree from scratch** in pure Python/NumPy using **Entropy** and **Information Gain**.
6. Evaluate our scratch implementation and compare its output to scikit-learn.
7. Connect Decision Trees to post-processing routing pipelines in computer vision.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from collections import Counter

# Set seed for reproducibility
np.random.seed(42)

## 1. Case Study Data Generation

We generate 90 samples of bounding boxes with two features:
1.  `area` (in pixels, ranging from 2,000 to 30,000)
2.  `aspect_ratio` (width / height, ranging from 0.5 to 2.5)

Our classes:
*   Class 0 (`valve`): Area > 15,000 and Aspect Ratio > 1.2
*   Class 1 (`flange`): Area > 15,000 and Aspect Ratio <= 1.2
*   Class 2 (`small-fitting`): Area <= 15,000

In [ ]:
m = 90

# Generate random features
areas = np.random.rand(m, 1) * 28000 + 2000
aspect_ratios = np.random.rand(m, 1) * 2.0 + 0.5
X = np.hstack((areas, aspect_ratios))

# Assign labels based on rules + some noise
y = np.zeros(m, dtype=int)
for i in range(m):
    area, aspect = X[i, 0], X[i, 1]
    if area <= 15000:
        y[i] = 2  # Small fitting
    elif aspect > 1.2:
        y[i] = 0  # Valve
    else:
        y[i] = 1  # Flange

# Plot the dataset
plt.figure(figsize=(8, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='blue', label='Class 0: Valve', alpha=0.7)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='red', label='Class 1: Flange', alpha=0.7)
plt.scatter(X[y == 2, 0], X[y == 2, 1], color='green', label='Class 2: Small Fitting', alpha=0.7)
plt.axvline(15000, color='black', linestyle='--', alpha=0.5, label='Area Threshold')
plt.axhline(1.2, color='gray', linestyle='--', alpha=0.5, label='Aspect Ratio Threshold')
plt.xlabel('Area (pixels)')
plt.ylabel('Aspect Ratio')
plt.title('PTT Object Classification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Decision Tree using Scikit-Learn

Let's train a scikit-learn classifier with `max_depth=3`.

In [ ]:
# Train model
clf = DecisionTreeClassifier(max_depth=3, criterion='entropy')
clf.fit(X, y)

# Evaluate training accuracy
y_pred_sklearn = clf.predict(X)
print(f"Training Accuracy: {accuracy_score(y, y_pred_sklearn) * 100:.2f}%")

Let's visualize the learned decision tree structure using `plot_tree`.

In [ ]:
plt.figure(figsize=(12, 8))
plot_tree(clf, feature_names=['area', 'aspect_ratio'], class_names=['Valve', 'Flange', 'Small Fitting'], filled=True, rounded=True)
plt.title('Learned Decision Tree Diagram')
plt.show()

## 3. Decision Tree from Scratch (Entropy and Information Gain)

Let's build a Decision Tree from scratch using recursive node splitting.
For each split, we select the feature and split threshold that maximizes the **Information Gain**:
$$IG(S, A) = H(S) - \left( \frac{|S_{\text{left}}|}{|S|} H(S_{\text{left}}) + \frac{|S_{\text{right}}|}{|S|} H(S_{\text{right}}) \right)$$

Where $H(S)$ is the entropy of dataset $S$:
$$H(S) = -\sum p_i \log_2 p_i$$

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       # Index of feature to split on
        self.threshold = threshold   # Threshold value for split
        self.left = left             # Left child Node
        self.right = right           # Right child Node
        self.value = value           # Class label if leaf node

    def is_leaf(self):
        return self.value is not None

class CustomDecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.root = None

    def _entropy(self, y):
        counts = np.bincount(y)
        probs = counts / len(y)
        return -np.sum([p * np.log2(p) for p in probs if p > 0])

    def _information_gain(self, y, X_col, threshold):
        parent_entropy = self._entropy(y)
        
        left_idx = np.where(X_col <= threshold)[0]
        right_idx = np.where(X_col > threshold)[0]
        
        if len(left_idx) == 0 or len(right_idx) == 0:
            return 0
            
        n = len(y)
        n_l, n_r = len(left_idx), len(right_idx)
        e_l, e_r = self._entropy(y[left_idx]), self._entropy(y[right_idx])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        
        return parent_entropy - child_entropy

    def _best_split(self, X, y):
        best_gain = -1
        split_idx, split_thresh = None, None
        m, n = X.shape
        
        for feature in range(n):
            X_col = X[:, feature]
            thresholds = np.unique(X_col)
            for threshold in thresholds:
                gain = self._information_gain(y, X_col, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature
                    split_thresh = threshold
                    
        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        m, n = X.shape
        n_labels = len(np.unique(y))
        
        if depth >= self.max_depth or n_labels == 1 or m < 2:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
            
        split_idx, split_thresh = self._best_split(X, y)
        if split_idx is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
            
        left_idx = np.where(X[:, split_idx] <= split_thresh)[0]
        right_idx = np.where(X[:, split_idx] > split_thresh)[0]
        
        left_child = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_child = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)
        
        return Node(feature=split_idx, threshold=split_thresh, left=left_child, right=right_child)

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_row(self, node, x):
        if node.is_leaf():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self._predict_row(node.left, x)
        return self._predict_row(node.right, x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])

# Train our custom model
tree_scratch = CustomDecisionTree(max_depth=3)
tree_scratch.fit(X, y)

y_pred_scratch = tree_scratch.predict(X)
print(f"Scratch Decision Tree Accuracy: {accuracy_score(y, y_pred_scratch) * 100:.2f}%")

## 4. Visualizing the Decision Boundaries

Let's see how our Custom Decision Tree partitions the feature space into rectangular subdivisions.

In [ ]:
from matplotlib.colors import ListedColormap

# Generate grid points
x_min, x_max = X[:, 0].min() - 2000, X[:, 0].max() + 2000
y_min, y_max = X[:, 1].min() - 0.2, X[:, 1].max() + 0.2
xx, yy = np.meshgrid(np.arange(x_min, x_max, 100),
                     np.arange(y_min, y_max, 0.01))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Predict on grid using our custom tree
Z = tree_scratch.predict(grid_points)
Z = Z.reshape(xx.shape)

# Plot decision boundary partitions
plt.figure(figsize=(10, 6))
cmap_light = ListedColormap(['#AAAAFF', '#FFAAAA', '#AAFFAA'])
cmap_bold = ['blue', 'red', 'green']

plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.5)
for c in [0, 1, 2]:
    plt.scatter(X[y == c, 0], X[y == c, 1], color=cmap_bold[c], label=f'Class {c}', edgecolor='k')

# Draw the splits identified by the custom tree
root_feat = tree_scratch.root.feature
root_thresh = tree_scratch.root.threshold
feat_name = 'Area' if root_feat == 0 else 'Aspect Ratio'
print(f"Root Split: {feat_name} <= {root_thresh:.2f}")

plt.xlabel('Area')
plt.ylabel('Aspect Ratio')
plt.title('Custom Decision Tree Partition Boundaries')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 💡 Connection to Computer Vision & YOLO
*   **Bounding Box Post-processing Routing:** Decision Trees are extremely useful in building fast heuristic rules. If you deploy a YOLO model to run on a low-power edge processor (like a Raspberry Pi or micro-controller), evaluating a complex neural network is expensive. You can run YOLO to get simple coordinate coordinates, and then run a lightweight **Decision Tree** script to quickly route predictions, filter background noise (e.g. `Area <= 15000`), or perform secondary label checking without needing another neural network.